# Step 2c — libpostal rule-based baseline
Rule (`dumb_libpostal_model.py`): parse both strings with libpostal, then a pair matches only if every check passes — house number, road, ZIP5. Each check has a setting (`road` strict/loose, `zip` strict/off, `number` strict/loose). Settings were compared on val; the two reported settings (strict/strict/strict and loose/off/loose, one per trade-off) were then checked once on test. All 8 combinations are shown on both splits, but only those two are reported. Nothing is trained. Results and explorations: **Diagnostics** (last cell).

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import itertools
import os

import pandas as pd
pd.set_option("display.width", 250)              # widen the line
pd.set_option("display.expand_frame_repr", False)  # never wrap, scroll sideways

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices, find_random_sample_given_mask
from model_baseline.dumb_libpostal_model import libpostal_baseline, _compare_parsed_addrs, _create_parsed_addresses, _get_all_libpost_baseline_options, _get_libpostal_results, get_pct_based_on_rules, LIBPOSTAL_FEATURES

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load data
Both datasets in the shared pair schema (`helpers/pairs.py`): `address_a/b`, `zip_a/b`, `neg_type`, … — Overture from `Overture/02_processing_overture_data.ipynb`, Stamford from `Stamford/03_convert_to_shared_schema.py`.

In [57]:
ot_train, ot_train_label = load_pairs("overture", "train")
ot_val, ot_val_label = load_pairs("overture", "val")
ot_test, ot_test_label = load_pairs("overture", "test")

In [58]:
# Same columns as Overture (Stamford/03_convert_to_shared_schema.py), so no adapter is needed
train_df, train_df_label = load_pairs("stamford", "train")
val_df, val_df_label = load_pairs("stamford", "val")
test_df, test_df_label = load_pairs("stamford", "test")

## 2. Rule settings + known-answer test
The test checks `_compare_parsed_addrs` on 13 hand-made pairs under every combination (expected values confirmed against libpostal's own parse/dedupe output). Run it before trusting any grid.

In [60]:
# Known-answer test: expected result per case for every rule combination

COMBOS = list(_get_all_libpost_baseline_options())   # (road, zip, number), same order as the columns
TEST_CASES = [
    # name,                    addr_a,                addr_b,              zip_a,        zip_b,   expected per combo, in COMBOS order
    ('identical',              '100 HAWLEY LN',       '100 HAWLEY LN',     '06905',      '06905', [True , True , True , True , True , True , True , True ]),
    ('different number',       '100 HAWLEY LN',       '200 HAWLEY LN',     '06905',      '06905', [False, False, False, False, False, False, False, False]),
    ('number on one side',     'HAWLEY LN',           '100 HAWLEY LN',     '06905',      '06905', [False, True , False, True , False, True , False, True ]),
    ('number suffix 509/509A', '509 MAIN ST',         '509A MAIN ST',      '06905',      '06905', [False, False, False, False, False, False, False, False]),
    ('number range 83-85/83',  '83-85 MAIN ST',       '83 MAIN ST',        '06905',      '06905', [False, False, False, False, False, False, False, False]),
    ('suffix abbreviation',    '100 HAWLEY LN',       '100 HAWLEY LANE',   '06905',      '06905', [True , True , True , True , True , True , True , True ]),
    ('directional N/S',        '100 N MAIN ST',       '100 S MAIN ST',     '06905',      '06905', [False, False, False, False, True , True , True , True ]),
    ('different road',         '100 MAIN ST',         '100 ELM ST',        '06905',      '06905', [False, False, False, False, False, False, False, False]),
    ('road typo',              '100 CONNECTICUT AVE', '100 CONNETICUT AVE','06905',      '06905', [False, False, False, False, False, False, False, False]),
    ('road parse failure',     'HAWLEY LN',           'HAWLEY LANE',       '06905',      '06905', [True , True , True , True , True , True , True , True ]),
    ('different ZIP',          '100 HAWLEY LN',       '100 HAWLEY LN',     '06905',      '06830', [False, False, True , True , False, False, True , True ]),
    ('ZIP+4 vs ZIP5',          '100 HAWLEY LN',       '100 HAWLEY LN',     '06905-1234', '06905', [True , True , True , True , True , True , True , True ]),
    ('ZIP missing one side',   '100 HAWLEY LN',       '100 HAWLEY LN',     None,         '06905', [True , True , True , True , True , True , True , True ]),
]

for name, addr_a, addr_b, zip_a, zip_b, expected in  TEST_CASES:
    for (road, zip_rule, number), correct_res in zip(COMBOS, expected):
    
        result = _compare_parsed_addrs(
            _create_parsed_addresses(addr_a),
            _create_parsed_addresses(addr_b),
            zip_a,
            zip_b,
            road,
            zip_rule,
            number,
        )
        # `is` so a None/1 result fails instead of passing as truthy
        assert result is correct_res, f"{name}: {addr_a} | {addr_b} | {zip_a} | {zip_b} | road({road}), zip({zip_rule}), number({number}). Expected {correct_res}, got {result!r}."


## 3. Overture — val vs test

In [62]:
# Overture: val vs test for all 8 rule combinations (predictions cached in libpostal_results/)
all_val, all_test = [], []
for road, zip_rule, number in _get_all_libpost_baseline_options():
    # Get results from libpostal or stored parquet
    val_preds = _get_libpostal_results(ot_val, road, zip_rule, number, "val")
    test_preds = _get_libpostal_results(ot_test, road, zip_rule, number, "test")
    #Get aggregate given typical mask
    table_val = review_typical_slices(ot_val, val_preds, ot_val_label)
    table_test = review_typical_slices(ot_test, test_preds, ot_test_label)
    #Additional features, so we can -groupby later
    for table in [table_val, table_test]:
        table["road_rule"] = road
        table["zip_rule"] = zip_rule
        table["number_rule"] = number
        table["pct_correct_float"] = table["% correct"].str[:-1].astype(float)
    
    all_val.append(table_val); all_test.append(table_test)

ret = get_pct_based_on_rules([pd.concat(all_val), pd.concat(all_test)], LIBPOSTAL_FEATURES, isolate_zip = None, isolate_number = None, suffixes = ["_val", "_test"])
ret["avg_diff_btw_sets"] = (ret["avg_pct_correct_float_val"] - ret["avg_pct_correct_float_test"]).abs()
# road_rule/zip_rule/number_rule/slice are the index, so they print on the left without being selected
out = ret[["avg_pct_correct_val", "avg_pct_correct_test"]].assign(
    avg_diff_btw_sets=ret["avg_diff_btw_sets"].map(lambda v: f"{v:.1f}%"))
print(out.to_string())

                                                                        avg_pct_correct_val avg_pct_correct_test avg_diff_btw_sets
road_rule zip_rule number_rule slice                                                                                              
loose     off      loose       neg: case1 (same street, diff number)                  98.3%                97.9%              0.4%
                               neg: case2 (same number, similar street)               50.0%                73.3%             23.3%
                               neg: case3 (easy)                                      99.4%                98.9%              0.5%
                               overall (accuracy)                                     98.6%                98.2%              0.4%
                               positives: identical                                  100.0%               100.0%              0.0%
                               positives: non-identical                            

## 4. Stamford — val vs test

In [65]:
# Stamford: val vs test for all 8 rule combinations (predictions cached in libpostal_results/)
all_val, all_test = [], []
for road, zip_rule, number in _get_all_libpost_baseline_options():
    # Get results from libpostal or stored parquet
    val_preds = _get_libpostal_results(val_df, road, zip_rule, number, "val", dataset="Stamford")
    test_preds = _get_libpostal_results(test_df, road, zip_rule, number, "test", dataset="Stamford")
    #Get aggregate given typical mask
    table_val = review_typical_slices(val_df, val_preds, val_df_label)
    table_test = review_typical_slices(test_df, test_preds, test_df_label)
    #Additional features, so we can -groupby later
    for table in [table_val, table_test]:
        table["road_rule"] = road
        table["zip_rule"] = zip_rule
        table["number_rule"] = number
        table["pct_correct_float"] = table["% correct"].str[:-1].astype(float)
    
    all_val.append(table_val); all_test.append(table_test)

ret = get_pct_based_on_rules([pd.concat(all_val), pd.concat(all_test)], LIBPOSTAL_FEATURES, isolate_zip = None, isolate_number = None, suffixes = ["_val", "_test"])
ret["avg_diff_btw_sets"] = (ret["avg_pct_correct_float_val"] - ret["avg_pct_correct_float_test"]).abs()
# road_rule/zip_rule/number_rule/slice are the index, so they print on the left without being selected
out = ret[["avg_pct_correct_val", "avg_pct_correct_test"]].assign(
    avg_diff_btw_sets=ret["avg_diff_btw_sets"].map(lambda v: f"{v:.1f}%"))
print(out.to_string())

                                                        avg_pct_correct_val avg_pct_correct_test avg_diff_btw_sets
road_rule zip_rule number_rule slice                                                                              
loose     off      loose       overall (accuracy)                     93.4%                94.0%              0.6%
                               positives: identical                  100.0%               100.0%              0.0%
                               positives: non-identical               36.0%                39.0%              3.0%
                   strict      overall (accuracy)                     93.8%                94.0%              0.2%
                               positives: identical                  100.0%               100.0%              0.0%
                               positives: non-identical               36.0%                39.0%              3.0%
          strict   loose       overall (accuracy)                     93.9%     

## Diagnostics
One-off checks for Step 2c (val phase 2026-10-14, test phase 2026-10-15), each as **question → result → query**. Settings are written `road/zip/number`. Unless noted, run by David in this notebook; entries marked *(Claude)* were run by Claude and the query is here to re-run.

*Note (2026-10-15):* both datasets now load in one shared schema. Stamford gained a `neg_type` (derived from the clean source rows), so its slice tables now include case1/case2/case3 and `find_random_sample_given_mask` mask numbers shift (Stamford mask 2 is now case1, not overall). The tables below were made before that and show Stamford without negative slices; every prediction is unchanged (same strings, verified).

### A. Results

**A1. Overture val — all 8 combinations (% correct per slice).** n: identical pos 3,390 · non-identical pos 797 · case1 13,781 · case2 34 · easy 23,863.

| road/zip/number | pos identical | pos non-identical | case1 | case2 | easy | overall |
|---|---|---|---|---|---|---|
| strict/strict/strict | 99.0 | 56.5 | 100.0 | 88.2 | 100.0 | 99.1 |
| strict/strict/loose | 99.0 | 67.4 | 98.3 | 88.2 | 100.0 | 98.7 |
| strict/off/strict | 100.0 | 57.2 | 100.0 | 88.2 | 100.0 | 99.2 |
| strict/off/loose | 100.0 | 68.8 | 98.3 | 88.2 | 99.4 | 98.5 |
| loose/strict/strict | 99.0 | 62.6 | 100.0 | 50.0 | 100.0 | 99.2 |
| loose/strict/loose | 99.0 | 73.9 | 98.3 | 50.0 | 100.0 | 98.8 |
| loose/off/strict | 100.0 | 63.7 | 100.0 | 50.0 | 100.0 | 99.3 |
| loose/off/loose | 100.0 | **75.7** | 98.3 | 50.0 | 99.4 | 98.6 |

Reference (Step 1b, default models, same val): non-identical positives LR 81.8 / RF 85.4 / XGB 84.9 — every libpostal setting is below LR on the slice that matters. No single best setting: each setting trades positive recall for rejection on one negative slice (see A3). Table rebuilt from the cached predictions *(Claude)*:
```python
rows = {}
for r, z, n in _get_all_libpost_baseline_options():
    p = pd.read_parquet(f"libpostal_results/{r}_{z}_{n}_Overture_val_results.1.parquet")
    rows[f"{r}/{z}/{n}"] = review_typical_slices(ot_val, p, ot_val_label).set_index("slice")["% correct"]
pd.DataFrame(rows).T
```

**A2. Stamford val — all 8 combinations** (n = 977: 97 positives, 880 negatives; Stamford has no `neg_type`, so negatives only show inside overall). Identical positives 11/11 and non-identical positives 31/86 (**36.0%**) under every setting; overall accuracy 93.4–94.4% (best strict/strict/strict 94.4, worst loose/off/loose 93.4) — the settings move only a handful of negatives. Query: the §4 cell.

**A3. What each setting does (Overture, from A1).**

| Setting | Effect | Size |
|---|---|---|
| road strict → loose | non-identical positives ↑, case2 rejection ↓ | +6–7 pts / 88.2 → 50.0 (13 of 34 rows) |
| zip strict → off | recovers identical positives with differing ZIPs; a few easy negatives get through when number is loose | 3,356 → 3,390 (+34) / easy 100.0 → 99.4 |
| number strict → loose | non-identical positives ↑, case1 rejection ↓ | +11–12 pts / 100.0 → 98.3 (~230 rows) |

### B. Predictions vs outcomes (David's hypotheses)

| Hypothesis | Outcome |
|---|---|
| road strict vs loose won't matter, because ZIP resolves most confusion | **Refuted on Overture** (A3: case2 88.2 vs 50.0, non-identical 67.4 vs 73.9 at zip strict/number loose). ZIP can't separate case1/case2 — they are ZIP-blocked by construction. On Stamford road changes accuracy by ≤ 0.3 pts. |
| ZIP will have a minor effect because most rows share a ZIP | Effect is small but real (A3: +34 identical positives with zip off). The supporting check (C1) doesn't test the claim yet — needs a per-slice split. |
| libpostal is better on the real data than on Stamford, because it can't handle arbitrary character edits | Consistent with the numbers: non-identical positive recall ~57–76% (Overture) vs 36.0% (Stamford), where Stamford's corruption is character-level edits on the street name. Spot check of the misses not done yet (open, D). |

### C. Explorations

**C1. Do most Overture val pairs share a ZIP?** 45.1% of all val rows have an identical full ZIP string. (An earlier version printed 100% — `len()` of the boolean Series counts every row; fixed by filtering first.) Overall is the wrong denominator — it is mostly easy negatives across towns; still to do: per slice, comparing `.str[:5]`.
```python
n = len(ot_val)
len(ot_val[ot_val["zip_a"] == ot_val["zip_b"]]) * 100 / n
```

**C2. What usually differs between the two addresses?** A random 10-row sample of `ot_val` — mostly unrelated easy negatives (90% of val is negatives), so it says little about positives. Use `find_random_sample_given_mask(ot_val, preds["pred"].values, ot_val_label, 1)` for non-identical positives instead.
```python
ot_val[["address_a", "address_b"]].sample(frac=1).head(10)
```

**C3. Does Stamford ever lack a house number?** No: 0.0% of val rows have a missing number on either side; 70.6% have different numbers on the two sides (mostly negatives).
```python
len(val_df[val_df["Address_a"].isna() | val_df["Address_b"].isna()]) * 100 / len(val_df)
len(val_df[val_df["Address_a"] != val_df["Address_b"]]) * 100 / len(val_df)
```

**C4. Known-answer test.** All 13 cases × 8 combinations pass (§2). Side finding *(Claude)*: `is_house_number_duplicate` returned only EXACT or NON on every pair tried (`509`/`509A`, `83-85`/`83`, `100`/`100 1/2`, `9d`/`9 d`, `12`/`twelve` → EXACT, `100`/`0100` → NON), so number:loose effectively differs from strict only on "number on one side". And `HAWLEY LANE` alone parses with no `road`, so the road check is skipped (a parse failure counts as a pass).

**C5. Reproduction check: does the default setting reproduce the first run?** loose/strict/strict now gives 99.0 / 62.6 / 100.0 / 50.0 / 100.0 vs the first run's 99.0 / 66.9 / 98.7 / 44.1 / ~100 *(Claude, 2026-10-14)*. Difference: the first run tried every `expand_address` expansion pair (`any()`); the current rule parses the raw strings once. Dropping expansion cost ~34 non-identical positives, and also accepted fewer case1/case2 negatives.

**C6. Why did identical-string positives fail in the first run?** *(Claude)* All 34 have different 5-digit ZIPs — the same 34 that zip:off recovers in A1.
```python
pos, ident = ot_val_label == 1, ot_val["address_a"] == ot_val["address_b"]
e = ot_val[pos & ident & (preds["pred"].values == 0)]
(e["zip_a"].str[:5] != e["zip_b"].str[:5]).sum(), len(e)
```

**C7. What causes the non-identical positive misses (first run, loose road, with expansion, 264 misses)?** *(Claude)* 116 number on one side only · 38 both numbers present but different (`509`/`509A`, `83-85`/`83`, `708`/`708A`) · 27 ZIP differs · rest mostly road NON on typos (`CONNECTICUT`/`CONNETICUT`). Categories overlap slightly.
```python
from postal.parser import parse_address
def num(s): return dict((c, v) for v, c in parse_address(s, language="en", country="us")).get("house_number")
ne = ot_val[pos & ~ident & (preds["pred"].values == 0)]
na, nb = ne["address_a"].map(num), ne["address_b"].map(num)
(na.isna() ^ nb.isna()).sum(), (na.notna() & nb.notna() & (na != nb)).sum(), (ne["zip_a"].str[:5] != ne["zip_b"].str[:5]).sum()
```

**C8. Effect of the Stamford adapter bug** *(Claude)*. Feeding street-only strings (no house number) gave loose/strict/strict accuracy 80.9% with 133 negatives accepted; with the house number it is 94.2% with 2 accepted — positive recall barely moves (44.3 → 43.3%). The first Stamford grid (all ~37% / 78–81%) was from the broken strings.

### E. Test phase (single run, settings fixed beforehand)

Reported settings, decided on val before test was run: **strict/strict/strict** (rejection end) and **loose/off/loose** (recall end). All 8 combinations were also run on test (§3/§4), for the val-vs-test check only — the reported pair stays fixed.

**E1. Reported results — test (val in brackets), % correct.**

| Dataset | Setting | pos identical | pos non-identical | case1 | case2 | easy | overall |
|---|---|---|---|---|---|---|---|
| Overture | strict/strict/strict | 99.1 (99.0) | **52.3** (56.5) | 100.0 (100.0) | 80.0 (88.2) | 100.0 (100.0) | 99.0 (99.1) |
| Overture | loose/off/loose | 100.0 (100.0) | **71.9** (75.7) | 97.9 (98.3) | 73.3 (50.0) | 98.9 (99.4) | 98.2 (98.6) |
| Stamford | strict/strict/strict | 100.0 (100.0) | **39.0** (36.0) | — | — | — | 94.8 (94.4) |
| Stamford | loose/off/loose | 100.0 (100.0) | **39.0** (36.0) | — | — | — | 94.0 (93.4) |

Test n — Overture: identical pos 3,543 · non-identical pos 843 · case1 14,448 · case2 15 · easy 24,995. Stamford: identical pos 17 · non-identical pos 59 · 766 rows. Query: §3 / §4 cells (val and test side by side, all 8 settings).

**E2. Are the Overture test results valid? (David: yes.)** Every slice except case2 is within ~1 pt of val, apart from non-identical positives (below). Case2 swings 8–30 pts between val and test — noise: test has 15 rows (1 row = 6.7 pts), val 34, and most case2 rows are **mirrored pairs** (the same two addresses in both orders, e.g. `220 CHAPEL RD` / `220 HALE RD`, `21 BRADLEY RD` / `21 S BRADLEY RD`), so ~17 distinct pairs in val, each pair getting one prediction twice. Several are probably label noise (same building: `100 TRAP FALLS RD EXT` / `ROAD EXT`, `900 MAIN ST S` / `… BLDG 2`). Case2 can't support a claim either way. **Non-identical positives are 3–5.5 pts lower on test under all 8 settings.** Not overfitting from choosing settings on val (the unchosen settings drop as much) → a property of the test ZIPs (different places, slightly harder matches). Report the test numbers, with that sentence.
```python
len(ot_val[ot_val["neg_type"] == "case2"]), len(ot_test[ot_test["neg_type"] == "case2"])   # 34, 15
preds = libpostal_baseline(ot_val, "loose", "strict", "loose")
find_random_sample_given_mask(ot_val, preds, ot_val_label, 3, ["address_a", "address_b", "zip_a", "zip_b"])   # same with ot_test
```

**E3. Are the Stamford test results valid? (David: yes, no surprises.)** Non-identical positives +3.0 pts on test (23/59 vs 31/86) — noise: 1 row = 1.7 pts and sampling wobble at n ≈ 60–90 is about ±5 pts. The +3.0 repeating under all 8 settings is not extra evidence: the positive slices don't change with the setting on Stamford, so it is one measurement printed 8 times. A random sample (mask 2 = overall, so mostly negatives) showed one positive miss, `227 DUNDEE RD` / `227 DUDEE RD` rejected — fits the character-edit hypothesis (B), but one row doesn't settle it. 48.1% (val) / 46.6% (test) of Stamford pairs have different ZIPs.
```python
preds = libpostal_baseline(val_df, "loose", "strict", "loose")
find_random_sample_given_mask(val_df, preds, val_df_label, 2, ["address_a", "address_b", "zip_a", "zip_b"])   # same with test_df
len(val_df[val_df["zip_a"] != val_df["zip_b"]]) * 100 / len(val_df)
```

### D. Still open
- Results row + one paragraph (finish threshold). Open choice: compare with Step 1b (LR/RF/XGB) on **val only** (both sides exist), or also run Step 1b's deferred single test run so every model has a test row.
- C1 per slice (positives / case1 / case2 / easy, `.str[:5]`).
- Spot check of Stamford non-identical misses (mask 1, not mask 2): are they street-name character edits returning NON?
- Stamford number:strict vs loose differ by 3 negatives despite 0% missing numbers (C3) → libpostal must sometimes fail to parse a number that is in the string. Unchecked.